# VoiceForge on a Colab T4
Select **Runtime → Change runtime type → T4 GPU**. Run the cells in order. The final cell plays audio served through VoiceForge.

In [ ]:
import subprocess
subprocess.run(["nvidia-smi", "-L"], check=True)
%pip -q install uv

In [ ]:
%%bash
set -euo pipefail
uv venv --allow-existing --python 3.12 /content/voiceforge-omni
uv pip install --python /content/voiceforge-omni/bin/python --torch-backend=cu129 \
  'vllm @ https://github.com/vllm-project/vllm/releases/download/v0.30.0/vllm-0.30.0%2Bcu129-cp38-abi3-manylinux_2_28_x86_64.whl' \
  'vllm-omni==0.30.0'
/content/voiceforge-omni/bin/python -c 'import torch; print(torch.cuda.get_device_name())'
/content/voiceforge-omni/bin/vllm --version
/content/voiceforge-omni/bin/python - <<'PY'
from pathlib import Path
import sysconfig

# vLLM-Omni 0.30.0 can return FP32 MTP embeddings on the T4 FP16 path.
path = Path(sysconfig.get_path("purelib")) / "vllm_omni/worker_v2/model_states/omni_model_state.py"
original = "embeds.index_copy_(0, batch_offsets, new_emb[:bsz].reshape(bsz, -1))"
patched = "embeds.index_copy_(0, batch_offsets, new_emb[:bsz].reshape(bsz, -1).to(dtype=embeds.dtype))"
source = path.read_text()
if source.count(original) == 1:
    path.write_text(source.replace(original, patched, 1))
elif source.count(patched) != 1:
    raise RuntimeError("Unexpected vLLM-Omni source. Check the installed version.")
print("vLLM-Omni T4 dtype workaround is active.")
PY
if [[ ! -d /content/voiceforge-smoke/.git ]]; then
  git clone https://github.com/dne-labs/Voice-inference.git /content/voiceforge-smoke
fi
cd /content/voiceforge-smoke
git fetch origin 5b48a31b2d6b25ab004b9541923a955173ef3056
git checkout --detach 5b48a31b2d6b25ab004b9541923a955173ef3056
cd apps/api
npm install --no-audit --no-fund

In [ ]:
import os
import time
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError

def start(command, name, **options):
    previous = globals().get(name)
    if previous is not None and previous.poll() is None:
        previous.terminate()
        previous.wait(timeout=30)
    log = Path(f"/content/{name}.log")
    with log.open("w") as output:
        return subprocess.Popen(command, stdout=output, stderr=subprocess.STDOUT, **options)

model = "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"
omni = start(["/content/voiceforge-omni/bin/vllm", "serve", model, "--omni",
              "--host", "127.0.0.1", "--port", "8090", "--dtype", "half",
              "--enforce-eager"], "omni")
deadline = time.monotonic() + 600
while time.monotonic() < deadline and omni.poll() is None:
    try:
        with urlopen("http://127.0.0.1:8090/v1/models", timeout=2):
            break
    except (URLError, TimeoutError):
        time.sleep(3)
else:
    print(Path("/content/omni.log").read_text(errors="replace")[-8000:])
    raise RuntimeError("Model server did not become ready. See /content/omni.log.")
gateway = start(["node", "server.js"], "gateway", cwd="/content/voiceforge-smoke/apps/api",
                env={**os.environ, "VLLM_OMNI_URL": "http://127.0.0.1:8090", "PORT": "3001",
                     "VLLM_OMNI_TIMEOUT_MS": "240000"})
time.sleep(2)
if gateway.poll() is not None:
    raise RuntimeError(Path("/content/gateway.log").read_text())
print("Model and gateway started.")

In [ ]:
import json
import wave
from urllib.request import Request
from urllib.error import URLError
from IPython.display import Audio, display

request = Request("http://127.0.0.1:3001/v1/speech",
                  data=json.dumps({"backend": "vllm", "text": "Hello from VoiceForge.",
                                   "voice_id": "ryan"}).encode(),
                  headers={"Content-Type": "application/json"})
path = Path("/content/voiceforge-vllm.wav")
try:
    with urlopen(request, timeout=255) as response:
        path.write_bytes(response.read())
except (URLError, TimeoutError) as error:
    if hasattr(error, "read"):
        print(error.read().decode(errors="replace"))
    for log in ("gateway", "omni"):
        log_path = Path(f"/content/{log}.log")
        if log_path.exists():
            print(f"--- {log}.log ---")
            print(log_path.read_text(errors="replace")[-8000:])
    raise
with wave.open(str(path)) as wav:
    if not wav.getnframes():
        raise RuntimeError("The WAV contains no audio.")
    print(f"{wav.getframerate()} Hz, {wav.getnframes() / wav.getframerate():.2f} seconds")
display(Audio(filename=str(path)))